# 👥 Mini-Project 2: Customer Insights Dashboard

**Objective**: Extract insights from customer data using descriptive statistics, group aggregations, and advanced visualization.

**Skills Tested**:
- Pandas groupby and pivot tables
- Descriptive Statistics
- Seaborn advanced plots (violins, heatmaps)
- Business insight extraction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

sns.set_theme(style="whitegrid", palette="muted")

## 1. Generate Customer Data
Run the cell below to create the synthetic CRM dataset.

In [ ]:
np.random.seed(101)
n_customers = 1000

df_customers = pd.DataFrame({
    'CustomerID': range(1, n_customers + 1),
    'Age': np.random.normal(loc=40, scale=12, size=n_customers).astype(int),
    'Income': np.random.lognormal(mean=11.0, sigma=0.6, size=n_customers),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], n_customers, p=[0.2, 0.3, 0.25, 0.25]),
    'TotalPurchases': np.random.poisson(lam=5, size=n_customers),
    'TotalSpend': 0.0
})

# Ensure logical constraints
df_customers['Age'] = np.clip(df_customers['Age'], 18, 90)
# Correlate spend with income and purchases
df_customers['TotalSpend'] = (df_customers['TotalPurchases'] * np.random.uniform(20, 100, n_customers)) + (df_customers['Income'] * 0.01)

df_customers.head()

## 2. Descriptive Statistics
Calculate central tendency and dispersion for Income and Age.

In [ ]:
# View statistical summary
print(df_customers[['Age', 'Income', 'TotalSpend']].describe())

# Notice the massive difference between Mean and Median (50%) for Income due to the lognormal distribution (Right Skew)
print("\nSkewness:\n", df_customers[['Age', 'Income', 'TotalSpend']].skew())

## 3. Customer Segmentation
Create age brackets and income tiers using `pd.cut` and `pd.qcut`.

In [ ]:
# Create Age Brackets (Fixed intervals)
df_customers['Age_Group'] = pd.cut(df_customers['Age'], 
                                   bins=[17, 30, 45, 60, 100],
                                   labels=['18-30', '31-45', '46-60', '60+'])

# Create Income Tiers (Equal sized quantiles)
df_customers['Income_Tier'] = pd.qcut(df_customers['Income'], 
                                      q=3, 
                                      labels=['Low', 'Medium', 'High'])

df_customers.head()

## 4. Visual Dashboard
Build a 2x2 grid of visualizations.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Income Distribution (Showing the right skew)
sns.histplot(df_customers['Income'], bins=50, kde=True, ax=axes[0, 0], color="skyblue")
axes[0, 0].set_title("Income Distribution (Right Skewed)")

# Plot 2: Spend by Age Group and Region
sns.boxplot(data=df_customers, x='Age_Group', y='TotalSpend', hue='Region', ax=axes[0, 1])
axes[0, 1].set_title("Total Spend by Age Group and Region")

# Plot 3: Scatter plot of Income vs Spend
sns.scatterplot(data=df_customers, x='Income', y='TotalSpend', hue='Income_Tier', alpha=0.6, ax=axes[1, 0])
axes[1, 0].set_xscale('log') # Use log scale for x-axis due to skew
axes[1, 0].set_title("Income vs Spend (Log Scale)")

# Plot 4: Correlation Heatmap
corr = df_customers[['Age', 'Income', 'TotalPurchases', 'TotalSpend']].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=axes[1, 1])
axes[1, 1].set_title("Feature Correlation Heatmap")

plt.tight_layout()
plt.show()

## 5. Business Insight Extraction
Using pivot tables to find actionable insights.

In [ ]:
# Pivot table showing average spend per Age Group and Income Tier
insight_table = pd.pivot_table(df_customers, 
                               values='TotalSpend', 
                               index='Age_Group', 
                               columns='Income_Tier', 
                               aggfunc='mean').round(2)
print("Average Spend by Demographic:\n")
print(insight_table)

# Heatmap of the pivot table
plt.figure(figsize=(8, 6))
sns.heatmap(insight_table, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Average Spend Heatmap")
plt.show()